# 02 - Data Cleaning and Preparation

## Project: Enterprise IT RAG-Based AI Document Assistant

### Objective

The objective of this notebook is to clean, standardize,
and validate the enterprise IT document dataset.

### Key Activities

1. Load the original raw dataset
2. Create a separate working copy
3. Standardize column names and metadata
4. Handle missing values
5. Identify and manage duplicate records
6. Standardize categorical values
7. Evaluate document quality and relevance
8. Validate the cleaned dataset
9. Export the cleaned dataset

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
current_path = Path.cwd()

project_root = None

for path in [current_path, *current_path.parents]:
    if (path / "data" / "raw").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError("Project directory not found.")

data_dir = project_root / "data" / "raw"

excel_files = list(data_dir.glob("*.xlsx"))

if len(excel_files) != 1:
    raise ValueError(f"Expected one Excel file, found {len(excel_files)}.")

file_path = excel_files[0]

raw_df = pd.read_excel(
    file_path,
    sheet_name="Document_Corpus"
)

print("Dataset loaded successfully!")
print("Dataset shape:", raw_df.shape)

Dataset loaded successfully!
Dataset shape: (84, 14)


In [3]:
clean_df = raw_df.copy(deep=True)

print("Working copy created successfully!")

print("Original dataset shape:", raw_df.shape)
print("Working dataset shape:", clean_df.shape)

Working copy created successfully!
Original dataset shape: (84, 14)
Working dataset shape: (84, 14)


In [4]:
clean_df.columns.tolist()

['document_id',
 'title',
 'category',
 'department',
 'owner',
 'effective_date',
 'review_date',
 'version',
 'status',
 'source_system',
 'access_level',
 'document_text',
 'quality_flags',
 'expected_action']

In [5]:
clean_df.dtypes

document_id            str
title                  str
category               str
department             str
owner                  str
effective_date         str
review_date            str
version            float64
status                 str
source_system          str
access_level           str
document_text          str
quality_flags          str
expected_action        str
dtype: object

In [6]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   document_id      84 non-null     str    
 1   title            84 non-null     str    
 2   category         84 non-null     str    
 3   department       84 non-null     str    
 4   owner            77 non-null     str    
 5   effective_date   83 non-null     str    
 6   review_date      77 non-null     str    
 7   version          82 non-null     float64
 8   status           84 non-null     str    
 9   source_system    84 non-null     str    
 10  access_level     84 non-null     str    
 11  document_text    84 non-null     str    
 12  quality_flags    84 non-null     str    
 13  expected_action  84 non-null     str    
dtypes: float64(1), str(13)
memory usage: 9.3 KB


In [7]:
clean_df.head()

,document_id,title,category,department,owner,effective_date,review_date,version,status,source_system,access_level,document_text,quality_flags,expected_action
0,DOC-0001,Privileged Access Standard,Access Management,Security,Security Owner,2024-01-24,2025-01-23,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,none,retain
1,DOC-0002,Privileged Access Standard - Working Copy 2,Access Management,Security,Security Owner,2024-02-02,2025-02-01,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,none,retain
2,DOC-0003,Privileged Access Standard - Working Copy 3,Access Management,Security,Security Owner,2024-02-11,2025-02-10,3.0,Superseded,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,none,retain
3,DOC-0004,Joiner-Mover-Leaver Procedure,Access Management,IT Operations,ITOperations Owner,2024-02-20,2025-02-19,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it operations teams and authorized service owners. I...,none,retain
4,DOC-0005,Joiner-Mover-Leaver Procedure - Working Copy 2,Access Mgmt,IT Operations,ITOperations Owner,2024-02-29,2025-02-28,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it operations teams and authorized service owners. I...,inconsistent_category,retain


In [8]:
clean_df.isnull().sum()

document_id        0
title              0
category           0
department         0
owner              7
effective_date     1
review_date        7
version            2
status             0
source_system      0
access_level       0
document_text      0
quality_flags      0
expected_action    0
dtype: int64

In [9]:

column_profile = pd.DataFrame({
    "column_name": clean_df.columns,
    "data_type": clean_df.dtypes.astype(str).values,
    "missing_count": clean_df.isnull().sum().values,
    "missing_percentage": (
        clean_df.isnull().mean().values * 100
    ).round(2),
    "unique_values": clean_df.nunique(dropna=True).values,
    "duplicate_count": [
        clean_df[column].duplicated().sum()
        for column in clean_df.columns
    ],
    "sample_value": [
        clean_df[column].dropna().iloc[0]
        if clean_df[column].notna().any()
        else None
        for column in clean_df.columns
    ]
})

column_profile


,column_name,data_type,missing_count,missing_percentage,unique_values,duplicate_count,sample_value
0,document_id,str,0,0.00,84,0,DOC-0001
1,title,str,0,0.00,82,2,Privileged Access Standard
2,category,str,0,0.00,19,65,Access Management
3,department,str,0,0.00,14,70,Security
4,owner,str,7,8.33,19,64,Security Owner
5,effective_date,str,1,1.19,83,0,2024-01-24
6,review_date,str,7,8.33,77,6,2025-01-23
7,version,float64,2,2.38,8,75,1.1
8,status,str,0,0.00,8,76,Approved
9,source_system,str,0,0.00,11,73,Controlled Knowledge Repository


In [10]:

report_dir = project_root / "reports" / "data_quality_report"
report_dir.mkdir(parents=True, exist_ok=True)

column_profile.to_csv(
    report_dir / "column_profile_before_cleaning.csv",
    index=False
)

print("Column profile saved successfully!")
print("Location:", report_dir)


Column profile saved successfully!
Location: C:\Users\main\Desktop\Projects\RAG Based AI Assistant\reports\data_quality_report


In [11]:

missing_columns = clean_df.columns[clean_df.isnull().any()].tolist()

print("Columns containing missing values:")
print(missing_columns)

missing_records = clean_df.loc[
    clean_df[missing_columns].notna().sum(axis=1) < len(missing_columns)
].copy()

missing_records


Columns containing missing values:
['owner', 'effective_date', 'review_date', 'version']


,document_id,title,category,department,owner,effective_date,review_date,version,status,source_system,access_level,document_text,quality_flags,expected_action
7,DOC-0008,Service Account Governance - Working Copy 2,Access Management,Security,NaN,2024-03-27,2025-03-27,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,missing_owner,retain
11,DOC-0012,Incident Priority and Escalation Matrix - Working Copy 3,Incident Management,IT Service Desk,ITServiceDesk Owner,2024-05-02,NaN,3.0,Superseded,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it service desk teams and authorized service owners....,missing_review_date,retain
26,DOC-0027,Vendor Support Escalation Guide - Working Copy 3,Service Levels,Vendor Management,NaN,2024-09-14,2025-09-14,3.0,Superseded,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to vendor management teams and authorized service owner...,missing_owner,retain
34,DOC-0035,Change Freeze Calendar Guidance - Working Copy 2,Change Management,Application Support,ApplicationSupport Owner,2024-11-25,NaN,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to application support teams and authorized service own...,missing_review_date,retain
45,DOC-0046,Cloud Resource Provisioning Standard,Network and Cloud,Cloud Platform,NaN,2025-03-04,2026-03-04,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to cloud platform teams and authorized service owners. ...,missing_owner,retain
57,DOC-0058,Secure File Transfer Procedure,Data Protection,Security,Security Owner,2025-06-20,NaN,1.1,Approved,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to security teams and authorized service owners. It des...,missing_review_date,retain
64,DOC-0065,Managed Laptop Troubleshooting Guide - Working Copy 2,End User Support,IT Service Desk,NaN,2025-08-22,2026-08-22,2.2,Draft,Controlled Knowledge Repository,Internal,Purpose and scope: This document applies to it service desk teams and authorized service owners....,missing_owner,retain
72,DOC-X001,Privileged Access Standard,Access Mgmt,IT Ops,NaN,NaN,NaN,NaN,Active,SharePoint export,Internal,Privileged access can be granted by any team member without manager or security approval. This s...,inaccurate_content; inconsistent_category; missing_owner; missing_dates; missing_version; mislea...,quarantine
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Platform,Cloud Platform,Cloud Owner,2025-07-01,NaN,2.1,Approved,Cloud Wiki,Internal,"Cloud resources must use approved landing zones, identity controls, encryption, logging, tagging...",missing_review_date; partial_content,retain_with_metadata_warning
79,DOC-X008,Password reset quick note,End User Support,IT Service Desk,NaN,2025-08-01,NaN,NaN,Published,Ticket comment,Internal,"If a user cannot sign in, verify identity using the approved process and follow the identity pla...",missing_owner; missing_review_date; missing_version; weak_provenance,retain_with_warning


In [12]:

duplicate_titles = clean_df[
    clean_df["title"].duplicated(keep=False)
].sort_values("title")

print("Number of records with repeated titles:", len(duplicate_titles))

duplicate_titles[
    ["document_id", "title", "category", "department", "version", "status"]
]


Number of records with repeated titles: 4


,document_id,title,category,department,version,status
45,DOC-0046,Cloud Resource Provisioning Standard,Network and Cloud,Cloud Platform,1.1,Approved
78,DOC-X007,Cloud Resource Provisioning Standard,Cloud Platform,Cloud Platform,2.1,Approved
0,DOC-0001,Privileged Access Standard,Access Management,Security,1.1,Approved
72,DOC-X001,Privileged Access Standard,Access Mgmt,IT Ops,NaN,Active
